## 1. Options

In [ ]:
# Configuration parameters for tile generation
SCALE = 1250                # Scale factor for printed model
TILE_SIZE = 0.2             # Size of physical printed tiles [meters]

# Magnet and base tile settings (used by Blender post-processing)
MAGNET_RADIUS = 5            # Magnet hole radius [mm]
MAGNET_HEIGHT = 2            # Magnet pocket depth [mm]
MAGNET_PERIMETER_OFFSET = 8  # Distance form tile edge to magnet centre [mm]
BASE_TILE_THICKNESS = 20     # Thickness of base tile [mm]
ANCHOR_Z_LEVEL = 0           # Z level for anchor points (used for alignment) [m]

# Buildings crossing the model edge are included whole and cut at the edge.
# The city is meshed this far beyond the model bounds so such buildings are
# complete; increase it if long buildings at the edge come out missing.
BUILDING_MARGIN = 50         # [m] (not used when ALIGN = True)

# Manually provided model bounds (SWEREF99)
MANUAL_BOUNDS = False        # Provide origin coords and area size manually (specify below)
ORIGIN_X = 319_470.0         # SW corner
ORIGIN_Y = 6_397_410.0       # SW corner
SIZE_X = 750                 # Size of real world bounding box [meters]
SIZE_Y = 1250                # Size of real world bounding box [meters]

# Aligning to existing model grid and conforming boundaries to old mesh
ALIGN = False                     # Set 'True' to align to existing model grid (specify below)
ALIGNMENT_NODE_X = 319_470.0      # SW node for alignment [319_470.0]
ALIGNMENT_NODE_Y = 6_397_410.0    # SW node for alignment [6_397_410.0]

## 2. Setup

In [ ]:
import importlib.util
import subprocess
import sys

# dtcc-core is installed from GitHub and its API changes often, so it is pinned
# to a specific commit. Update this hash deliberately and re-test the notebook.
DTCC_CORE_COMMIT = "f8ab79b7b589759a7f9279eac576a725f722ed6a"  # 2026-11-02

REQUIRED_PACKAGES = {
    "shapely":    "shapely",
    "trimesh":    "trimesh",
    "geopandas":  "geopandas",
    f"git+https://github.com/dtcc-platform/dtcc-core.git@{DTCC_CORE_COMMIT}": "dtcc_core",
    "ipyleaflet": "ipyleaflet",
    "ipywidgets": "ipywidgets",
}

missing = [pkg for pkg, module in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print(f"Installing missing packages: {', '.join(missing)}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print("All required packages are already installed.")

from pathlib import Path
import numpy as np
import math
import matplotlib.pyplot as plt

import dtcc_core as dtcc
from dtcc_core.model import GeometryType, City
from dtcc_core.model.geometry.bounds import Bounds
from dtcc_core.io.data.wrapper import download_footprints, download_pointcloud
from dtcc_core.io.meshes import load_mesh
from dtcc_core.builder.meshing import conform_boundary, SurfaceMeshClipper
from dtcc_core.builder import (
    clean_footprints,
    extract_roof_points,
    compute_building_heights,
    build_terrain_raster,
)
import trimesh
import geopandas as gpd
from shapely.geometry import Polygon, box
from ipyleaflet import Map, DrawControl, basemaps
from ipywidgets import Output
from IPython.display import display
import shutil, os

# --- Install Blender (Colab only) --------------------------------------------

BLENDER_VER = "4.1.1"

if shutil.which("blender") is None:
    if sys.platform == "linux":
        BLENDER_URL = f"https://download.blender.org/release/Blender4.1/blender-{BLENDER_VER}-linux-x64.tar.xz"
        subprocess.run(["wget", "-q", BLENDER_URL, "-O", "/content/blender.tar.xz"], check=True)
        subprocess.run(["tar", "-xf", "/content/blender.tar.xz", "-C", "/content"], check=True)
        os.symlink(f"/content/blender-{BLENDER_VER}-linux-x64/blender", "/usr/local/bin/blender")
    else:
        print(f"Blender not found on PATH. Please install Blender {BLENDER_VER} and ensure it is on your PATH.")
else:
    print(f"Blender found: {shutil.which('blender')}")

# --- Other inputs, change only if necessary ---------------------------------

RASTER_CELL_SIZE = 2.0
RASTER_RADIUS    = 3.0
OUTLIER_STDDEV   = 3.0
MAX_MESH_SIZE    = 5.0
MIN_MESH_ANGLE   = 25.0
SMOOTHING        = 3

MERGE_MAX_DIST   = 0.5   # Footprints closer than this are merged [m]
MERGE_MIN_AREA   = 10.0  # Buildings smaller than this are dropped [m2]
MIN_HEIGHT       = 3.0   # Minimum building height [m]

TILES_DIR      = Path("raw_tiles")
OUT_DIR        = Path("processed_tiles")
OLD_MESHES_DIR = Path("old_meshes")
for d in (TILES_DIR, OUT_DIR, OLD_MESHES_DIR):
    d.mkdir(parents=True, exist_ok=True)


# 1) --- Interactive map ------------------------------------------------------

def select_area():
    output = Output()

    map_widget = Map(basemap=basemaps.OpenStreetMap.Mapnik, center=(57.687845516523026, 11.976528661251745), zoom=13)
    map_widget.layout.width  = "1000px"
    map_widget.layout.height = "600px"

    draw_control = DrawControl()
    draw_control.rectangle    = {
        "shapeOptions": {"color": "#0000FF", "fillOpacity": 0.3},
        "showArea": True,
        "metric": True,
    }
    draw_control.polygon      = {
        "shapeOptions": {"color": "#FF0000", "fillOpacity": 0.3},
        "showArea": True,
        "metric": True,
    }
    draw_control.polyline     = {}
    draw_control.circle       = {}
    draw_control.circlemarker = {}

    import pyproj
    from shapely.geometry import shape
    from shapely.ops import transform as shp_transform

    project = pyproj.Transformer.from_crs("EPSG:4326", "EPSG:3006", always_xy=True).transform

    def handle_draw(target, action, geo_json):
        if action == 'created' and geo_json['geometry']['type'] == 'Polygon':
            coords = geo_json['geometry']['coordinates'][0]
            is_rectangle = len(coords) == 5

            geom_sweref = shp_transform(project, shape(geo_json['geometry']))
            minx, miny, maxx, maxy = geom_sweref.bounds
            w = maxx - minx
            h = maxy - miny

            BOUNDARY_GEOJSON.append({
                "type": "Feature",
                "geometry": geo_json['geometry'],
                "properties": {"shape": "rectangle" if is_rectangle else "polygon"}
            })

            shape_label = "Rectangle" if is_rectangle else "Polygon"
            shape_num   = len(BOUNDARY_GEOJSON)

            with output:
                print(f"─── Shape {shape_num} ({shape_label}) ───────────────────")
                print(f"  E–W width : {w:>8.0f} m")
                print(f"  N–S height: {h:>8.0f} m")
                print(f"  Area      : {geom_sweref.area/1e6:>8.4f} km²")

    draw_control.on_draw(handle_draw)
    map_widget.add_control(draw_control)

    if MANUAL_BOUNDS:
        print("Draw shapes around buildings to be excluded")
    else:
        print("1. Draw shape defining model bounds")
        print("2. Draw shapes around buildings to be excluded")
    print("-" * 50)
    display(map_widget, output)


# 2) --- Convert geojson ------------------------------------------------------

def convert_geojson(boundary_geojson):
    boundary   = []
    exclusions = []
    if len(boundary_geojson) > 0:
        gdf = gpd.GeoDataFrame.from_features(boundary_geojson, crs="EPSG:4326").to_crs(3006)
        boundary   = gdf.geometry.iloc[0]
        exclusions = gdf.geometry if MANUAL_BOUNDS else gdf.geometry.iloc[1:]
    return boundary, exclusions


# 3) --- Compute bounds -------------------------------------------------------

def compute_bounds(boundary):
    print('Computing area bounds…')
    grid_size = TILE_SIZE * SCALE

    if MANUAL_BOUNDS:
        origin_x, origin_y, size_x, size_y = ORIGIN_X, ORIGIN_Y, SIZE_X, SIZE_Y
    else:
        minx, miny, maxx, maxy = boundary.bounds
        size_x, size_y = maxx - minx, maxy - miny
        origin_x, origin_y = round(minx), round(miny)

    if ALIGN:
        min_x = ALIGNMENT_NODE_X + math.floor((origin_x - ALIGNMENT_NODE_X) / grid_size) * grid_size
        min_y = ALIGNMENT_NODE_Y + math.floor((origin_y - ALIGNMENT_NODE_Y) / grid_size) * grid_size
    else:
        min_x, min_y = origin_x, origin_y

    max_x = min_x + math.ceil(((origin_x + size_x - min_x) / grid_size)) * grid_size
    max_y = min_y + math.ceil(((origin_y + size_y - min_y) / grid_size)) * grid_size

    bounds = Bounds(min_x, min_y, max_x, max_y)
    n_tiles_x = (max_x - min_x) / grid_size
    n_tiles_y = (max_y - min_y) / grid_size

    print(f'Model Size: {n_tiles_x * TILE_SIZE:.2f} x {n_tiles_y * TILE_SIZE:.2f} m ({n_tiles_x:.0f}x{n_tiles_y:.0f} tiles)')
    print(f'Min X: {min_x}  Min Y: {min_y}  Max X: {max_x}  Max Y: {max_y}')
    return bounds, grid_size, n_tiles_x, n_tiles_y, min_x, min_y, max_x, max_y


# 4) --- Load buildings -------------------------------------------------------

def mesh_bounds(bounds):
    """Model bounds expanded by BUILDING_MARGIN: the area that is meshed.

    With ALIGN the mesh edge must stay on the model edge so it can be
    conformed to the old meshes, so no margin is added.
    """
    m = 0 if ALIGN else BUILDING_MARGIN
    return Bounds(bounds.xmin - m, bounds.ymin - m, bounds.xmax + m, bounds.ymax + m)

def get_footprint(b):
    """2D footprint polygon of a building, or None if it has none.

    b.footprint() also finds footprints stored on building parts, which
    b.geometry[GeometryType.LOD0] does not.
    """
    surf = b.footprint()
    if surf is None:
        return None
    poly = surf.to_polygon()
    return None if poly.is_empty else poly


def load_buildings(bounds, min_x, max_x, min_y, max_y, grid_size, exclusions):
    boundary = box(min_x, min_y, max_x, max_y)
    fig, ax = plt.subplots(figsize=(8, 8), dpi=120)
    ax.axis('off')
    gpd.GeoSeries([boundary]).plot(ax=ax, color='none', edgecolor='black', linewidth=1.0)

    print("Loading building footprints…")
    # Download over the expanded area, then keep every building that touches
    # the model, including those crossing its edge.
    buildings = download_footprints(bounds=mesh_bounds(bounds), provider='dtcc')
    buildings = [b for b in buildings if get_footprint(b) is not None]
    # Merge footprints of the same building before exclusion, so an exclusion
    # shape removes whole buildings rather than some of their pieces.
    buildings = clean_footprints(
        buildings, min_building_detail=MERGE_MAX_DIST, min_area=MERGE_MIN_AREA
    )
    buildings = [b for b in buildings if get_footprint(b).intersects(boundary)]

    if len(exclusions) > 0:
        exclusion_union = exclusions.union_all()

        if not exclusion_union.intersects(boundary):
            print('Exclusion area is outside of model bounds')

        gpd.GeoSeries(exclusion_union).plot(ax=ax, color='none', edgecolor='red', linewidth=0.5)

        b_kept     = []
        kept_fp    = []
        removed_fp = []

        for b in buildings:
            fp = get_footprint(b)
            if exclusion_union.covers(fp):
                removed_fp.append(fp)
            else:
                b_kept.append(b)
                kept_fp.append(fp)

        if removed_fp:
            gpd.GeoSeries(removed_fp).plot(ax=ax, color='red')
        print(f'{len(removed_fp)} buildings excluded from mesh')

    else:
        b_kept  = buildings
        kept_fp = [get_footprint(b) for b in b_kept]

    gpd.GeoSeries(kept_fp).plot(ax=ax, color='lightgray', zorder=0)

    for x in np.arange(min_x, max_x + grid_size, grid_size):
        ax.plot([x, x], [min_y, max_y], color='black', linestyle='--', linewidth=0.5, zorder=10)
    for y in np.arange(min_y, max_y + grid_size, grid_size):
        ax.plot([min_x, max_x], [y, y], color='black', linestyle='--', linewidth=0.5, zorder=10)

    ax.set_xlim(min_x, max_x)
    ax.set_ylim(min_y, max_y)
    fig.savefig("layout_preview.png", dpi=300, bbox_inches="tight")

    return b_kept


# 5) --- Load pointcloud ------------------------------------------------------

def load_pointcloud(bounds):
    print("Downloading pointcloud…")
    return download_pointcloud(bounds=mesh_bounds(bounds))


# 6) --- Build terrain + building heights -------------------------------------

def make_terrain_and_buildings(pointcloud, buildings_all, bounds):
    print("Building terrain raster…")
    raster = build_terrain_raster(
        pointcloud,
        cell_size=RASTER_CELL_SIZE,
        radius=RASTER_RADIUS,
        ground_only=True,
    )

    print(f"Raster bounds: {raster.bounds}")

    print("Extracting roof points + computing heights…")

    # Filter buildings to those within the raster bounds
    rb = raster.bounds
    raster_box = box(rb.xmin, rb.ymin, rb.xmax, rb.ymax)
    buildings_in_raster = [
        b for b in buildings_all
        if raster_box.covers(get_footprint(b).centroid)
    ]
    print(f"Buildings within raster: {len(buildings_in_raster)}/{len(buildings_all)}")

    buildings = extract_roof_points(
        buildings_in_raster,
        pointcloud,
        statistical_outlier_remover=True,
        roof_outlier_neighbors=5,
        roof_outlier_margin=1.5,
    )
    # Heights are stored as estimated_height; buildings without usable roof
    # points get MIN_HEIGHT.
    buildings = compute_building_heights(
        buildings, raster, min_building_height=MIN_HEIGHT, overwrite=True
    )

    return raster, buildings


# 7) --- Build city + mesh ----------------------------------------------------

def make_mesh(raster, buildings, bounds):
    print("Creating city and meshing…")
    city = City()
    city.add_terrain(raster)
    city.add_buildings(buildings, remove_outside_terrain=False)

    # LOD1: footprints extruded to building height. Footprints closer than
    # MERGE_MAX_DIST are merged and buildings below MERGE_MIN_AREA dropped.
    mesh = city.build_surface_mesh(
        lod="lod1",
        min_building_detail=MERGE_MAX_DIST,
        min_building_area=MERGE_MIN_AREA,
        max_mesh_size=MAX_MESH_SIZE,
        min_mesh_angle=MIN_MESH_ANGLE,
        smoothing=SMOOTHING,
        pipeline_mode='strict',
    )

    if ALIGN:
        print("Conforming boundary to existing meshes…")
        old_meshes = [load_mesh(stl) for stl in OLD_MESHES_DIR.glob("*.stl")]
        mesh = conform_boundary(
            new_mesh=mesh,
            old_meshes=old_meshes,
            smoothing_iterations=5,
            tolerance=10.0,
        )

    # Save the mesh cut to the model bounds, so it can be used as an old mesh
    # for ALIGN when extending the model later.
    SurfaceMeshClipper(mesh).clip_to_bounds(bounds).save('raw_city_mesh.stl')
    return mesh


# 8) --- Tile, extrude, scale -------------------------------------------------

def generate_tiles(mesh, min_x, min_y, n_tiles_x, n_tiles_y):
    real_tile_size = TILE_SIZE * SCALE

    step   = BASE_TILE_THICKNESS / 1000 * SCALE
    anchor = ANCHOR_Z_LEVEL
    min_t  = (MAGNET_HEIGHT + MAGNET_RADIUS + 1) / 1000 * SCALE

    # Cut tiles on the model grid. The mesh extends beyond the model, so
    # buildings crossing the model edge are cut there like at any tile edge.
    clipper = SurfaceMeshClipper(mesh)
    nx, ny  = round(n_tiles_x), round(n_tiles_y)

    print('Cutting, extruding and scaling tiles...')

    for ix in range(nx):
        for iy in range(ny):
            row, col = ix + 1, iy + 1
            x0 = min_x + ix * real_tile_size
            y0 = min_y + iy * real_tile_size
            tile = clipper.clip_to_bounds(
                Bounds(x0, y0, x0 + real_tile_size, y0 + real_tile_size)
            )
            if len(tile.faces) == 0:
                print(f'Tile {row}_{col} is empty, skipped')
                continue
            tile.save(str(TILES_DIR / f'raw_tile_{row}_{col}.stl'))

            min_z = float(tile.vertices[:, 2].min())
            base_level = anchor + math.floor((min_z - min_t - anchor) / step) * step

            ext_tile = tile.create_printable_solid(base_z=base_level)
            ext_tile.save(str(OUT_DIR / f'tile_{row}_{col}.stl'))

            tri = trimesh.load(str(OUT_DIR / f'tile_{row}_{col}.stl'))
            tri.apply_scale(1.0 / SCALE)
            tri.apply_translation([-min_x / SCALE, -min_y / SCALE, 0])
            tri.export(str(OUT_DIR / f'tile_{row}_{col}.stl'))


## 3. Area selection

In [ ]:
# Select area boundary and buildings to exclude

BOUNDARY_GEOJSON = []
select_area()

## 4. Download buildings and preview Layout

In [ ]:
# --- Calculate bounds and load building footprints ---------------------------
boundary, exclusions = convert_geojson(BOUNDARY_GEOJSON)
bounds, grid_size, n_tiles_x, n_tiles_y, min_x, min_y, max_x, max_y = compute_bounds(boundary)
buildings_kept = load_buildings(bounds, min_x, max_x, min_y, max_y, grid_size, exclusions)

## 5. Generate mesh and process tiles

In [ ]:
# --- Generate and Save Mesh -------------------------------------------------
pointcloud = load_pointcloud(bounds)
raster, buildings = make_terrain_and_buildings(pointcloud, buildings_kept, bounds)
mesh = make_mesh(raster, buildings, bounds)
generate_tiles(mesh, min_x, min_y, n_tiles_x, n_tiles_y)

## 6. Generate magnet holes

In [ ]:
# --- Blender script override: same inputs, remove unused internals ----------
BLENDER_PROCESS_SCRIPT = r'''import addon_utils
import bpy, sys, os, time, glob
from mathutils import Vector


# enable STL add-on (provides import/export operators)
addon_utils.enable("io_mesh_stl", default_set=True, persistent=True)


def get_arg(name, default=None, cast=str):
    argv = sys.argv
    if "--" not in argv:
        return default
    argv = argv[argv.index("--") + 1:]
    if name not in argv:
        return default
    i = argv.index(name)
    if i + 1 >= len(argv):
        return default
    return cast(argv[i + 1])


# === Parameters ===

in_dir = get_arg("--in_dir", "./tiles_output", str)
out_dir = get_arg("--out_dir", "./tiles_processed", str)

# Keep old CLI inputs for compatibility

tile_size = get_arg("--tile_size", 0.20, float)
magnet_radius_mm = get_arg("--magnet_radius_mm", 5.0, float)
magnet_height_mm = get_arg("--magnet_height_mm", 2.0, float)
magnet_perimeter_offset = get_arg("--magnet_perimeter_offset_mm", 8.0, float)

# Convert mm to m
magnet_radius = magnet_radius_mm / 1000.0
magnet_height = magnet_height_mm / 1000.0

os.makedirs(out_dir, exist_ok=True)

# Discover all tile STL files in input directory
tile_files = sorted(glob.glob(os.path.join(in_dir, "*.stl")))
total_tiles = len(tile_files)

start_time = time.time()
print(f"Processing {total_tiles} tile(s) from {in_dir}", flush=True)


def add_corner_magnet_holes(tile_obj, tile_index):
    """Create 4 magnet hole cutters at the corners of the tile."""
    # Get tile mesh bounds to determine position
    b = [v.co for v in tile_obj.data.vertices]
    if not b:
        return None

    min_v = Vector((min(v[0] for v in b), min(v[1] for v in b), min(v[2] for v in b)))
    max_v = Vector((max(v[0] for v in b), max(v[1] for v in b), max(v[2] for v in b)))

    tile_center_x = (min_v.x + max_v.x) / 2.0
    tile_center_y = (min_v.y + max_v.y) / 2.0
    tile_bottom_z = min_v.z

    # Magnet hole cylinder parameters
    offset_m = magnet_perimeter_offset / 1000.0
    safety = 0.001
    effective_offset = max(offset_m, magnet_radius + safety)
    magnet_cylinder_extra = 0.0005
    magnet_cylinder_depth = magnet_height + magnet_cylinder_extra * 2
    magnet_z = tile_bottom_z + (magnet_height / 2)

    half = tile_size / 2.0
    dx = half - effective_offset
    dy = half - effective_offset

    corner_positions = [
        (tile_center_x - dx, tile_center_y - dy),
        (tile_center_x + dx, tile_center_y - dy),
        (tile_center_x - dx, tile_center_y + dy),
        (tile_center_x + dx, tile_center_y + dy),
    ]

    cutters = []

    for i, (cx, cy) in enumerate(corner_positions):
        # Magnet hole cylinder
        bpy.ops.mesh.primitive_cylinder_add(
            radius=magnet_radius,
            depth=magnet_cylinder_depth,
            location=(cx, cy, magnet_z)
        )
        cyl = bpy.context.active_object
        cyl.name = f"MagnetHole_tile_{tile_index}_{i}"
        cutters.append(cyl)

        # Countersink cone
        cone_base_radius = magnet_radius
        cone_height = cone_base_radius
        cone_base_z = tile_bottom_z + magnet_height
        cone_center_z = cone_base_z + cone_height / 2.0

        bpy.ops.mesh.primitive_cone_add(
            radius1=cone_base_radius,
            radius2=0.0,
            depth=cone_height,
            location=(cx, cy, cone_center_z),
            rotation=(0.0, 0.0, 0.0)
        )
        cone = bpy.context.active_object
        cone.name = f"MagnetCountersink_tile_{tile_index}_{i}"
        cutters.append(cone)

    # Union all cutters
    union_obj = cutters[0]
    for i, cutter in enumerate(cutters[1:], start=1):
        bpy.ops.object.select_all(action='DESELECT')
        union_obj.select_set(True)
        bpy.context.view_layer.objects.active = union_obj

        union_mod = union_obj.modifiers.new(name=f"Union_{i}", type='BOOLEAN')
        union_mod.object = cutter
        union_mod.operation = 'UNION'
        union_mod.solver = 'EXACT'

        bpy.ops.object.modifier_apply(modifier=union_mod.name)
        bpy.data.objects.remove(cutter, do_unlink=True)

    union_obj.name = f"MagnetCutters_tile_{tile_index}"
    return union_obj


# Main processing loop
for tile_index, tile_path in enumerate(tile_files, start=1):
    tile_start = time.time()
    tile_name = os.path.basename(tile_path)

    print(f"[Tile {tile_index}/{total_tiles}] Importing {tile_name}...", flush=True)

    # Clear scene
    bpy.ops.object.select_all(action='SELECT')
    bpy.ops.object.delete(use_global=False)

    # Import tile
    bpy.ops.import_mesh.stl(filepath=tile_path)
    tile_obj = bpy.context.selected_objects[0]
    tile_obj.name = f"Tile_{tile_index}"

    # Clean up mesh
    bpy.context.view_layer.objects.active = tile_obj
    bpy.ops.object.mode_set(mode='EDIT')
    bpy.ops.mesh.select_all(action='SELECT')
    bpy.ops.mesh.remove_doubles(threshold=0.0001)
    try:
        bpy.ops.mesh.fill_holes()
    except Exception:
        pass
    bpy.ops.mesh.normals_make_consistent(inside=False)
    bpy.ops.object.mode_set(mode='OBJECT')

    print(f"[Tile {tile_index}/{total_tiles}] Adding magnet holes...", flush=True)

    # Create magnet hole cutters
    magnet_cutter_obj = add_corner_magnet_holes(tile_obj, tile_index)

    if magnet_cutter_obj is not None:
        # Apply magnet holes as boolean difference
        mod_magnet = tile_obj.modifiers.new(name="MagnetCutters", type='BOOLEAN')
        mod_magnet.object = magnet_cutter_obj
        mod_magnet.operation = 'DIFFERENCE'
        mod_magnet.solver = 'EXACT'
        mod_magnet.use_self = True
        mod_magnet.use_hole_tolerant = True

        bpy.ops.object.select_all(action='DESELECT')
        bpy.context.view_layer.objects.active = tile_obj

        try:
            bpy.ops.object.modifier_apply(modifier="MagnetCutters")
        except RuntimeError:
            print(f"[Tile {tile_index}/{total_tiles}] WARNING: Failed to apply magnet cutters.", flush=True)

        # Delete cutter
        bpy.data.objects.remove(magnet_cutter_obj, do_unlink=True)

    # Export processed tile
    bpy.ops.object.select_all(action='DESELECT')
    tile_obj.select_set(True)
    export_path = os.path.join(out_dir, tile_name)
    bpy.ops.export_mesh.stl(filepath=export_path, use_selection=True)

    print(f"[Tile {tile_index}/{total_tiles}] Exported to {export_path} ({time.time() - tile_start:.0f} s)", flush=True)

    bpy.data.objects.remove(tile_obj, do_unlink=True)

elapsed_s = time.time() - start_time
print(f"Done. {total_tiles} tile(s) processed in {elapsed_s:.0f} s.", flush=True)
'''

import os as _os
blender_script_name = 'blender_script.py'
blender_script = _os.path.join(_os.getcwd(), blender_script_name)
tiles_input_dir = str(OUT_DIR.resolve())
tiles_output_dir = tiles_input_dir

# Write Blender script from notebook text
with open(blender_script, 'w', encoding='utf-8') as f:
    f.write(BLENDER_PROCESS_SCRIPT)

cmd = f'''blender -b -P "{blender_script}" -- \
  --in_dir "{tiles_input_dir}" \
  --out_dir "{tiles_output_dir}" \
  --tile_size {TILE_SIZE} \
  --magnet_radius_mm {MAGNET_RADIUS} \
  --magnet_height_mm {MAGNET_HEIGHT} \
  --magnet_perimeter_offset_mm {MAGNET_PERIMETER_OFFSET}
'''

!{cmd}